In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
print(f"train data shape :", train_data.shape)
print(f"test data shape :", test_data.shape)

train data shape : (2000, 8)
test data shape : (500, 7)


In [4]:
print(train_data.columns.tolist())
print(test_data.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']


In [5]:
#test_data["Prediction"] = "B C A"

In [6]:
#submission = test_data[["id", "Prediction"]].rename(columns={"id": "ID"})
#submission.to_csv("submission.csv", index=False)

In [7]:
#print(submission.head())

The above random sampling submission had given 0.35425

## Model - 1: EDA  (Lookup Model)

Identified duplicates, unique and did vlookup in excel with train and test data. Based on my observations, tried to predict using lookup between train and test data.

This is from scratch model without using any ML or pretained weights. Prediction is based on excel based analysis.

In [8]:
print("Duplicates in Prompts:", train_data['prompt'].duplicated().sum())
print("Duplicates in option A:", train_data['A'].duplicated().sum())
print("Duplicates in option B:", train_data['B'].duplicated().sum())
print("Duplicates in option C:", train_data['C'].duplicated().sum())
print("Duplicates in option D:", train_data['D'].duplicated().sum())
print("Duplicates in option E:", train_data['E'].duplicated().sum())
print("Overall Unique Rows:", len(train_data.drop_duplicates(subset=['prompt', 'A', 'B', 'C', 'D', 'E'])))


Duplicates in Prompts: 242
Duplicates in option A: 1684
Duplicates in option B: 1672
Duplicates in option C: 1697
Duplicates in option D: 1682
Duplicates in option E: 1680
Overall Unique Rows: 1817


**Observations:**

1. Prompts are less duplicated than Options. So options are recycled heavily
2. 183 prompts with same options are duplicated
3. Each options have too many duplicates, the range is 1672 to 1697 duplicates

In [9]:
print("Duplicates in Prompts:", test_data['prompt'].duplicated().sum())
print("Duplicates in option A:", test_data['A'].duplicated().sum())
print("Duplicates in option B:", test_data['B'].duplicated().sum())
print("Duplicates in option C:", test_data['C'].duplicated().sum())
print("Duplicates in option D:", test_data['D'].duplicated().sum())
print("Duplicates in option E:", test_data['E'].duplicated().sum())
print("Overall Unique Rows:", len(test_data.drop_duplicates(subset=['prompt', 'A', 'B', 'C', 'D', 'E'])))


Duplicates in Prompts: 8
Duplicates in option A: 261
Duplicates in option B: 256
Duplicates in option C: 263
Duplicates in option D: 264
Duplicates in option E: 257
Overall Unique Rows: 493


In [10]:
# Checking unique texts per column
all_options = ['A', 'B', 'C', 'D', 'E']

for opt in all_options:
    unique = train_data[opt].nunique()
    print(f"Unique Option {opt} texts: {unique}")

# Count unique texts across all columns combined
all_texts = pd.concat([train_data[opt] for opt in all_options])
print(f"\nTotal texts across all columns: {len(all_texts)}")
print(f"Unique texts across all columns: {all_texts.nunique()}")

Unique Option A texts: 316
Unique Option B texts: 328
Unique Option C texts: 303
Unique Option D texts: 318
Unique Option E texts: 320

Total texts across all columns: 10000
Unique texts across all columns: 1584


In [11]:
# Build lookup table
answer_text_lookup = {}

for _, row in train_data.iterrows():
    # Getting the correct answer
    correct_text = row[row['answer']]
    
    # Mapping answer to each options {'paris':'paris', 'berlin':'paris', 'london':'paris' etc.,}
    for opt in all_options:
        answer_text_lookup[row[opt]] = correct_text 

print(f"Total entries: {len(answer_text_lookup)}")

Total entries: 1584


In [12]:
matched = 0
for _, row in test_data.iterrows():
    for opt in all_options:
        if row[opt] in answer_text_lookup:  ## training data keys matching with test data options
            matched += 1

print(f"Total test options: {len(test_data)*5}")
print(f"Found in lookup:    {matched}")
print(f"Match %:            {matched/(len(test_data)*5)*100:.1f}%")

Total test options: 2500
Found in lookup:    2483
Match %:            99.3%


In [13]:
## Function for prediction

## If the test data option is available in lookup table, then pickup that option and compare 
## with correct answer. If the answer matches, score 9999 for the option or update the len of the character
## len of the char will be used for subsequent best answers for 2nd and 3rd positions.

def prediction_lookup(row):
    scores = {}
    
    for opt in all_options:
        opt_text = row[opt]
        
        # Check if this option text is in our lookup
        if opt_text in answer_text_lookup:
            correct_text = answer_text_lookup[opt_text]
            
            # Is it the correct answer?
            if opt_text == correct_text:
                scores[opt] = 9999  # correct answer
            else:
                scores[opt] = len(opt_text)  # length of the character to determine the answer
        else:
            scores[opt] = len(opt_text)
    
    # Rank by highest score first
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked

In [14]:
from sklearn.metrics import accuracy_score, f1_score

def mapk(actual, predicted, k=3):
    score = 0.0
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            score += 1.0 / (p.index(a) + 1)
    return score / len(actual)

In [15]:
# # Prediction on training data

# actual          = train_data['answer'].tolist()
# predicted_all   = train_data.apply(prediction_lookup, axis=1).tolist()
# predicted_top1  = [p[0] for p in predicted_all]

# # Score calc
# map3  = mapk(actual, predicted_all)
# acc   = accuracy_score(actual, predicted_top1)
# f1    = f1_score(actual, predicted_top1, average='weighted')

# print(f"MAP3    : {map3:.4f}")
# print(f"Accuracy : {acc:.4f}")
# print(f"F1 Score : {f1:.4f}")

In [16]:
## Prediction on Test data

# predictions = []

# for _, row in test_data.iterrows():
#     ranked = prediction_lookup(row)
#     predictions.append(' '.join(ranked[:3]))

# # Create submission file
# submission         = test_data[['id']].copy()
# submission.columns = ['ID']
# submission['Prediction'] = predictions

# submission.to_csv('/kaggle/working/submission.csv', index=False)

# print(f"Shape: {submission.shape}")
# print(submission.head(10))

In [17]:
## Logging in wandb

# from kaggle_secrets import UserSecretsClient
# import wandb

# secret = UserSecretsClient()
# wandb.login(key=secret.get_secret("WANDB_API_KEY"))

# # Logging Model 1 (lookup model)
# run = wandb.init(
#     entity="21f3001142-iyyiitm",
#     project="dlgenai-t226",
#     name="model-1-eda-lookup",
#     config={
#         "model"       : "EDA Lookup Table",
#         "approach"    : "lookup + char length fallback",
#         "train_size"  : len(train_data),
#         "test_size"   : len(test_data),
#         "lookup_size" : len(answer_text_lookup),})

# run.log({
#     "map3_train" : map3,   
#     "map3_test"  : 0.75187,  
#     "accuracy"    : acc,
#     "f1_score"    : f1,
# })

# run.finish()

## Conclusion on Model 1 — EDA Lookup Table (From Scratch)

- Random guess (B C A) prediction is 0.35
- Afeter exploring the data using Excel and the EDA discovery: found duplicate options in train and test
- Built lookup table using Python dictionary
- Lookup table prediction is 0.75187

## Model - 2: MiniLM Semantic Similarity (Pretrained Model)

**What is MiniLM?**
- Full name: sentence-transformers/all-MiniLM-L6-v2
- Pretrained model from Hugging Face
- Converts text into 384 dimensional vectors
- Understands MEANING of text (semantic similarity)

**Approach**
- Encode prompt and all 5 options using MiniLM
- Calculate cosine similarity between prompt and each option
- Rank options by similarity score
- Higher similarity = more likely correct answer

**Why MiniLM over TF-IDF?**
- TF-IDF only looks at word overlap
- MiniLM understands meaning
- "car" and "vehicle" = similar in MiniLM
- "car" and "vehicle" = 0 similarity in TF-IDF

In [18]:
from sentence_transformers import SentenceTransformer, util

# Loading miniLM
model_minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print(f"Embedding dimension: {model_minilm.get_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


In [19]:
## Ranking function

# def get_minilm_ranking(row):
#     # Convert prompt to 384 dimensional vector
#     prompt_emb = model_minilm.encode(row['prompt'], convert_to_tensor=True)
    
#     scores = {}
    
#     # Convert each option to 384 dimensional vector
#     for opt in all_options:
#         opt_emb    = model_minilm.encode(row[opt], convert_to_tensor=True)
#     # Calculate similarity between prompt and option
#         scores[opt] = util.cos_sim(prompt_emb, opt_emb).item()
    
#     # Ranking options highest similarity first
#     ranked = sorted(scores, key=scores.get, reverse=True)
#     return ranked

In [20]:
## Testing on first row the training data

# row1 = train_data.iloc[0]

# print("Prompt:", row1['prompt'][:80])
# print("Correct answer:", row1['answer'])
# print()

# result = get_minilm_ranking(row1)
# print("MiniLM ranking:", result)
# print("Top 3:", ' '.join(result[:3]))

# if result[0] == row1['answer']:
#     print("Correct Answer!")
# else:
#     print(f"Wrong Answer {row1['answer']} at position {result.index(row1['answer'])+1}")

**Observation:**
MiniLM ranks by semantic similarity to prompt that is Questions. But doesnt know the facts or reality.

In [21]:
# Prediction on all training data

# actual_m2      = train_data['answer'].tolist()
# predicted_m2   = train_data.apply(get_minilm_ranking, axis=1).tolist()
# predicted_top1_m2 = [p[0] for p in predicted_m2]

# map3_m2 = mapk(actual_m2, predicted_m2)
# acc_m2  = accuracy_score(actual_m2, predicted_top1_m2)
# f1_m2   = f1_score(actual_m2, predicted_top1_m2, average='weighted')

# print(f"\nModel 2 Results:")
# print(f"MAP3    : {map3_m2:.4f}")
# print(f"Accuracy : {acc_m2:.4f}")
# print(f"F1 Score : {f1_m2:.4f}")

In [22]:
# Prediction on Test Data

# predictions_m2 = []
# for _, row in test_data.iterrows():
#     ranked = get_minilm_ranking(row)
#     predictions_m2.append(' '.join(ranked[:3]))

# # Submission file creation
# submission_m2         = test_data[['id']].copy()
# submission_m2.columns = ['ID']
# submission_m2['Prediction'] = predictions_m2

# submission_m2.to_csv('/kaggle/working/submission.csv', index=False)

# print(f"Shape: {submission_m2.shape}")
# print(submission_m2.head(5))

In [23]:
## Logging in wandb

from kaggle_secrets import UserSecretsClient
import wandb

secret = UserSecretsClient()
wandb.login(key=secret.get_secret("WANDB_API_KEY"))

# # Logging Model 2 (Pretrained Model - miniLM)
# run = wandb.init(
#     entity="21f3001142-iyyiitm",
#     project="dlgenai-t226",
#     name="model-2-minilm",
#     config={
#         "model"         : "MiniLM",
#         "pretrained"    : "sentence-transformers/all-MiniLM-L6-v2",
#         "embedding_dim" : 384,
#         "approach"      : "cosine similarity",
#         "train_size"    : len(train_data),
#         "test_size"     : len(test_data),})

# run.log({
#     "map3_train" : map3_m2,
#     "map3_test"  : 0.38653,
#     "accuracy"    : acc_m2,
#     "f1_score"    : f1_m2,})

# run.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## Conclusion on Model 2 — miniLM (Pretrained Model)

- Random Guess (B C A) prediction was 0.35
- miniLM predicted 0.3865 which is close to random guess. This shows miniLM needs facts/reality to be built to predict more accurately. 
- This triggers us for RAG with miniLM

## Model - 3: RAG (MiniLM + External Data Source)

RAG:
Question → Search Knowledge Base → Get Relevant Facts

                                          ↓
    Question + Facts → MiniLM → Predict (with facts!)

**Approach**

Step 1: Choose external knowledge base. Decided to use MMLU as it has many MCQs similar to our data. MMLU is good for STEM, Philosophy, History and Economics MCQs.

Step 2: Encoding it using MiniLM embeddings

Step 3: Indexing using FAISS 

Step 4: For each question:
          
          - Search index for relevant facts
          
          - Get top 3 relevant passages

Step 5: Combine question + facts + options

Step 6: Use similarity to rank options

Step 7: Predict top 3 answers

## Loading MMLU Dataset

In [24]:
# from datasets import load_dataset

# mmlu = load_dataset("cais/mmlu", "all", split="test")
# print(f"MMLU size: {len(mmlu)}")
# print(f"Columns: {mmlu.column_names}")
# print(mmlu[0])

In [25]:
# # Create knowledge base from MMLU
# # Combining question + correct answer as knowledge text

# mmlu_data = pd.DataFrame(mmlu)

# def create_knowledge_base(row):
#     question    = row['question']
#     correct_ans = row['choices'][row['answer']]
#     subject     = row['subject']
    
#     # Format: "Subject: X. Question: Y. Answer: Z"
#     return f"Subject: {subject}. {question} Answer: {correct_ans}"

# mmlu_data['knowledge'] = mmlu_data.apply(create_knowledge_base, axis=1)

# print(f"Knowledge base size: {len(mmlu_data)}")
# print("\nSample knowledge text:")
# print(mmlu_data['knowledge'][0])
# print(mmlu_data['knowledge'][1])

## Encoding using miniLM

In [26]:
# print(f"Total texts to encode: {len(mmlu_data)}")

# # This creates 384 dimensions vector for each text
# knowledge_texts     = mmlu_data['knowledge'].tolist()
# knowledge_embeddings = model_minilm.encode(
#     knowledge_texts,
#     batch_size=64)         # encodes 64 texts at a time

# print(f"Embeddings shape: {knowledge_embeddings.shape}")

## Indexing using FAISS

In [27]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.5 MB/s eta 0:00:00


In [28]:
# import faiss

# # FAISS need float32, hence converting the encoded knowledge text
# embeddings_f32 = knowledge_embeddings.astype(np.float32)

# # Normalizing embeddings for cosine similarity
# faiss.normalize_L2(embeddings_f32)

# # Creating FAISS index
# dimension = 384  
# index     = faiss.IndexFlatIP(dimension) # cosine similarity 
# index.add(embeddings_f32)

# print(f"Total vectors in index: {index.ntotal}")

## Building Retrievel function

In [29]:
# def retrieve_context(question, top_k=3):
#     # Convert question to 384 dimension vector
#     question_emb = model_minilm.encode(
#         [question], 
#         convert_to_tensor=False
#     ).astype(np.float32)
    
#     # Normalize
#     faiss.normalize_L2(question_emb)
    
#     # Search FAISS index for top_k similar texts
#     scores, indices = index.search(question_emb, top_k)
    
#     # Return relevant knowledge texts
#     contexts = []
#     for idx in indices[0]:
#         contexts.append(mmlu_data['knowledge'].iloc[idx])
    
#     return contexts

In [30]:
# def predict_rag(row):
#     # Get the question
#     question = row['prompt']
    
#     # Retrieve top 3 relevant contexts
#     contexts = retrieve_context(question, top_k=3)
    
#     # Combine question + contexts
#     combined_context = question + " " + " ".join(contexts)
    
#     # Encode combined context
#     context_emb = model_minilm.encode(
#         combined_context, 
#         convert_to_tensor=True
#     )
    
#     # Compare each option against context
#     scores = {}
#     for opt in all_options:
#         opt_emb     = model_minilm.encode(row[opt], convert_to_tensor=True)
#         scores[opt] = util.cos_sim(context_emb, opt_emb).item()
    
#     # Rank options by similarity
#     ranked = sorted(scores, key=scores.get, reverse=True)
#     return ranked

In [31]:
# ## Testing the first row in training data set

# question = train_data['prompt'].iloc[0]
# print("Question:", question[:80])
# print()

# contexts = retrieve_context(question, top_k=3)
# print("Retrieved contexts:")
# for i, ctx in enumerate(contexts):
#     print(f"\nContext {i+1}: {ctx[:150]}")

In [32]:
# row1   = train_data.iloc[0]
# result = predict_rag(row1)

# print("Question:", row1['prompt'][:80])
# print("Correct answer:", row1['answer'])
# print("RAG prediction:", result[:3])

# if result[0] == row1['answer']:
#     print("Correct answer")
# else:
#     print(f"Wrong answer! Correct at position {result.index(row1['answer'])+1}")

In [33]:
## Predict on all training data

# actual_rag    = train_data['answer'].tolist()
# predicted_rag = train_data.apply(predict_rag, axis=1).tolist()
# predicted_top1_rag = [p[0] for p in predicted_rag]

# map3_rag = mapk(actual_rag, predicted_rag)
# acc_rag  = accuracy_score(actual_rag, predicted_top1_rag)
# f1_rag   = f1_score(actual_rag, predicted_top1_rag, average='weighted')

# print(f"\nModel 3 RAG Results:")
# print(f"MAP3    : {map3_rag:.4f}")
# print(f"Accuracy : {acc_rag:.4f}")
# print(f"F1 Score : {f1_rag:.4f}")

## Conclusion on Model 3 — RAG 

- In Model-2, miniLM predicted 0.3865. This shows miniLM needs facts/reality to be built to predict more accurately. 
- With miniLM + MMLU Data Source, the RAG provided little improvement over Model-2
| Metric         | MiniLM alone | RAG    |
|----------------|--------------|--------|
| Train MAP3    | 0.4231       | 0.4401  |
| Train Accuracy | 0.2610       | 0.2700 |
| Train F1       | 0.2631       | 0.2697 |
| Test MAP3     | 0.3865       | 0.4401 |

## Model - 4: RAG ( Lookup Model + External Data Source)

In [34]:
# # Combining RAG with lookup table, that is look up first, then fall back to RAG

# def predict_lookup_rag(row):
#     scores = {}
#     none_phrases = ['none of the above', 'all of the above',
#                     'both a and', 'none of these']

#     for opt in all_options:
#         opt_text = row[opt]
#         if opt_text in answer_text_lookup:
#             correct_text = answer_text_lookup[opt_text]
#             if opt_text == correct_text:
#                 scores[opt] = 9999  # confident match
#             else:
#                 scores[opt] = len(opt_text)  # char length
#         else:
#             scores[opt] = len(opt_text)  # char length
    
#     # Check if lookup is confident
#     if max(scores.values()) < 9999:
#         # No confident match → use RAG
#         rag_ranked = predict_rag(row)
#         for i, opt in enumerate(rag_ranked):
#             scores[opt] = len(all_options) - i  # rank score
    
#     ranked = sorted(scores, key=scores.get, reverse=True)
#     return ranked

# # Evaluate
# predicted_lookup_rag = train_data.apply(predict_lookup_rag, axis=1).tolist()
# map3_lr = mapk(actual_rag, predicted_lookup_rag)
# print(f"Lookup + RAG MAP3: {map3_lr:.4f}")

In [35]:
# ## Using Lookup + RAG to predict on Test data

# predictions_rag = []
# for _, row in test_data.iterrows():
#     ranked = predict_lookup_rag(row)
#     predictions_rag.append(' '.join(ranked[:3]))

# # Create submission
# submission_rag         = test_data[['id']].copy()
# submission_rag.columns = ['ID']
# submission_rag['Prediction'] = predictions_rag
# submission_rag.to_csv('/kaggle/working/submission.csv', index=False)

# print(f"Shape: {submission_rag.shape}")
# print(submission_rag.head(5))

In [36]:
# run = wandb.init(
#     entity="21f3001142-iyyiitm",
#     project="dlgenai-t226",
#     name="model-3-rag",
#     config={
#         "model"          : "RAG Pipeline",
#         "knowledge_base" : "MMLU",
#         "kb_size"        : len(mmlu_data),
#         "encoder"        : "MiniLM",
#         "retriever"      : "FAISS",
#         "top_k"          : 3,
#         "approach"       : "lookup + RAG fallback",
#     }
# )
# run.log({
#     "train_map3"    : map3_rag,
#     "train_accuracy" : acc_rag,
#     "train_f1"       : f1_rag,
#     "test_map3"     : 0.74979,
# })
# run.finish()

## Conclusion on Model 4 — Lookup + RAG 

### Approach
Combined both Lookup + RAG:
- Lookup table handles questions seen in training
- RAG handles unseen questions with external knowledge

### Results
| Model              | Test MAP3 |
|--------------------|------------|
| Lookup alone       | 0.75187    |
| MiniLM alone       | 0.38653    |
| RAG (MiniLM + external data source)  | 0.44010    |
| RAG (Lookup + external data source)       | 0.74979    |


Lookup Still Dominates!!!...So my thinking is i made mistake in choosing miniLM and also the external data source MMLU. Realized MMLU is more of Law related data than STEM, Philosopy and History. I will now use Bert model as it can contextualize the data more to predict. Then i will also use wikipedia as recommended by Gemini AI and ChatGPT. Felt wikipedia can provide cover STEM, Philosopy, History and will have all common data.


## Model - 5: DeBERTa

In [37]:
# from transformers import AutoTokenizer, AutoModel
# import torch

# # Check GPU available
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Device: {device}")

# # Load DeBERTa
# model_name = "microsoft/deberta-v3-large"
# tokenizer_deberta = AutoTokenizer.from_pretrained(model_name)
# model_deberta     = AutoModel.from_pretrained(model_name)
# model_deberta     = model_deberta.to(device)  # move to GPU
# model_deberta.eval()

# print(f"DeBERTa loaded on {device}!")

In [38]:
# def get_deberta_embedding(text, max_length=512):
#     inputs = tokenizer_deberta(
#         text,
#         return_tensors='pt',
#         max_length=max_length,
#         truncation=True,
#         padding=True
#     ).to(device)  # move inputs to GPU
    
#     with torch.no_grad():
#         outputs = model_deberta(**inputs)
    
#     cls_embedding = outputs.last_hidden_state[:, 0, :]
#     return cls_embedding

# print("DeBERTa embedding function created!")

In [39]:
# # Quick test
# test_text = "What is Heidegger's view on time?"
# emb = get_deberta_embedding(test_text)
# print(f"Embedding shape: {emb.shape}")
# # Should be [1, 1024] ← 1024 dimensions!

In [40]:
# def get_deberta_ranking(row):
#     # Step 1: Get prompt embedding
#     prompt_emb = get_deberta_embedding(row['prompt'])
    
#     scores = {}
    
#     # Step 2: Get each option embedding
#     for opt in all_options:
#         opt_emb = get_deberta_embedding(row[opt])
        
#         # Step 3: Cosine similarity
#         similarity = torch.nn.functional.cosine_similarity(
#             prompt_emb, opt_emb
#         )
#         scores[opt] = similarity.item()
    
#     # Step 4: Rank highest first
#     ranked = sorted(scores, key=scores.get, reverse=True)
#     return ranked

In [41]:
# all_options = ['A', 'B', 'C', 'D', 'E']

# row1   = train_data.iloc[0]
# result = get_deberta_ranking(row1)

# print("Prompt:", row1['prompt'][:80])
# print("Correct answer:", row1['answer'])
# print("DeBERTa ranking:", result)
# print("Top 3:", ' '.join(result[:3]))

# if result[0] == row1['answer']:
#     print("CORRECT!")
# else:
#     print(f"WRONG! Correct at position {result.index(row1['answer'])+1}")

In [42]:
# from sklearn.metrics import accuracy_score, f1_score

# actual_deb      = train_data['answer'].tolist()
# predicted_deb   = train_data.apply(
#                     get_deberta_ranking, axis=1).tolist()
# predicted_top1_deb = [p[0] for p in predicted_deb]

# map3_deb = mapk(actual_deb, predicted_deb)
# acc_deb  = accuracy_score(actual_deb, predicted_top1_deb)
# f1_deb   = f1_score(actual_deb, predicted_top1_deb, 
#                     average='weighted')

# print(f"\nModel 5 DeBERTa Results:")
# print(f"MAP3    : {map3_deb:.4f}")
# print(f"Accuracy : {acc_deb:.4f}")
# print(f"F1 Score : {f1_deb:.4f}")

In [43]:
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses
)
from torch.utils.data import DataLoader

# Prepare training examples
train_examples = []

for _, row in train_data.iterrows():
    question    = row['prompt']
    correct_ans = row[row['answer']]
    
    # Positive pair → correct answer
    train_examples.append(
        InputExample(
            texts=[question, correct_ans],
            label=1.0  # similar
        )
    )
    
    # Negative pairs → wrong answers
    for opt in all_options:
        if opt != row['answer']:
            train_examples.append(
                InputExample(
                    texts=[question, row[opt]],
                    label=0.0  # not similar
                )
            )

print(f"Total training examples: {len(train_examples)}")
# Expected: 2000 × 5 = 10000 examples

Total training examples: 10000


/tmp/ipykernel_23/1004828892.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


In [44]:
# Load fresh MiniLM for fine-tuning
model_ft = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

# Create dataloader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

# Loss function
train_loss = losses.CosineSimilarityLoss(model_ft)

# Fine-tune!
print("Fine-tuning MiniLM...")
model_ft.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True
)

print("Fine-tuning done!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Fine-tuning MiniLM...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.141502


Fine-tuning done!


In [45]:
# Cell 3 — Ranking function for fine-tuned MiniLM
def get_finetuned_ranking(row):
    # Use fine-tuned model instead of original
    prompt_emb = model_ft.encode(
        row['prompt'], convert_to_tensor=True)
    
    scores = {}
    for opt in all_options:
        opt_emb     = model_ft.encode(
            row[opt], convert_to_tensor=True)
        scores[opt] = util.cos_sim(prompt_emb, opt_emb).item()
    
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked



In [46]:

actual_ft      = train_data['answer'].tolist()
predicted_ft   = train_data.apply(
                    get_finetuned_ranking, axis=1).tolist()
predicted_top1_ft = [p[0] for p in predicted_ft]

map3_ft = mapk(actual_ft, predicted_ft)
acc_ft  = accuracy_score(actual_ft, predicted_top1_ft)
f1_ft   = f1_score(actual_ft, predicted_top1_ft,
                   average='weighted')

print(f"\nModel 6 Fine-tuned MiniLM Results:")
print(f"MAP@3    : {map3_ft:.4f}")
print(f"Accuracy : {acc_ft:.4f}")
print(f"F1 Score : {f1_ft:.4f}")


Model 6 Fine-tuned MiniLM Results:
MAP@3    : 0.9972
Accuracy : 0.9945
F1 Score : 0.9945


In [47]:
# Generate test predictions

# predictions_ft = []
# for _, row in test_data.iterrows():
#     ranked = get_finetuned_ranking(row)
#     predictions_ft.append(' '.join(ranked[:3]))

# submission_ft         = test_data[['id']].copy()
# submission_ft.columns = ['ID']
# submission_ft['Prediction'] = predictions_ft
# submission_ft.to_csv('/kaggle/working/submission.csv', index=False)

# print("Submission file created")
# print(submission_ft.head(5))

In [48]:
import wandb

# run = wandb.init(
#     entity="21f3001142-iyyiitm",
#     project="dlgenai-t226",
#     name="model-6-finetuned-minilm",
#     config={
#         "model"      : "Fine-tuned MiniLM",
#         "base_model" : "all-MiniLM-L6-v2",
#         "epochs"     : 3,
#         "batch_size" : 16,
#         "loss"       : "CosineSimilarityLoss",
#         "examples"   : 10000,
#     }
# )
# run.log({
#     "train_map@3"    : 0.9954,
#     "train_accuracy" : 0.9910,
#     "train_f1"       : 0.9910,
#     "test_map@3"     : 0.75020,
# })
# run.finish()


## Lookup and Fine Tuned MiniLM

In [49]:
# def predict_lookup_finetuned(row):
#     scores = {}
    
#     for opt in all_options:
#         opt_text = row[opt]
#         if opt_text in answer_text_lookup:
#             correct_text = answer_text_lookup[opt_text]
#             if opt_text == correct_text:
#                 scores[opt] = 9999
#             else:
#                 scores[opt] = len(opt_text)
#         else:
#             # Using fine-tuned MiniLM for unmatched!
#             prompt_emb  = model_ft.encode(
#                 row['prompt'], convert_to_tensor=True)
#             opt_emb     = model_ft.encode(
#                 opt_text, convert_to_tensor=True)
#             scores[opt] = util.cos_sim(
#                 prompt_emb, opt_emb).item()
    
#     ranked = sorted(scores, key=scores.get, reverse=True)
#     return ranked

# # Generate submission
# predictions_lft = []
# for _, row in test_data.iterrows():
#     ranked = predict_lookup_finetuned(row)
#     predictions_lft.append(' '.join(ranked[:3]))

# submission_lft         = test_data[['id']].copy()
# submission_lft.columns = ['ID']
# submission_lft['Prediction'] = predictions_lft
# submission_lft.to_csv('/kaggle/working/submission.csv', index=False)
# print("Lookup + Fine-tuned MiniLM submission saved!")

## RAG with Wikipedia (Better knowledge base)

In [50]:
from datasets import load_dataset

print("Loading Wikipedia dataset...")

# New way to load Wikipedia
wiki = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True  # stream instead of download all!
)

print("Wikipedia loaded in streaming mode!")

Loading Wikipedia dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia loaded in streaming mode!


In [51]:
# Filter relevant articles
relevant_keywords = [
    'physics', 'philosophy', 'mathematics',
    'chemistry', 'biology', 'quantum',
    'astronomy', 'engineering', 'science',
    'theorem', 'theory', 'molecule'
]

wiki_articles = []
count = 0

for article in wiki:
    # Check if article is relevant
    title_lower = article['title'].lower()
    if any(kw in title_lower for kw in relevant_keywords):
        wiki_articles.append({
            'title': article['title'],
            'text' : article['text'][:500]  # first 500 chars
        })
    
    count += 1
    if count % 10000 == 0:
        print(f"Processed {count} articles, "
              f"found {len(wiki_articles)} relevant")
    
    # Stop after finding 5000 relevant articles
    if len(wiki_articles) >= 5000:
        break

wiki_df = pd.DataFrame(wiki_articles)
print(f"Sample titles:")
print(wiki_df['title'].head(10).to_string())

Processed 10000 articles, found 61 relevant
Processed 20000 articles, found 98 relevant
Processed 30000 articles, found 128 relevant
Processed 40000 articles, found 162 relevant
Processed 50000 articles, found 196 relevant
Processed 60000 articles, found 216 relevant
Processed 70000 articles, found 250 relevant
Processed 80000 articles, found 278 relevant
Processed 90000 articles, found 327 relevant
Processed 100000 articles, found 369 relevant
Processed 110000 articles, found 415 relevant
Processed 120000 articles, found 446 relevant
Processed 130000 articles, found 494 relevant
Processed 140000 articles, found 535 relevant
Processed 150000 articles, found 576 relevant
Processed 160000 articles, found 618 relevant
Processed 170000 articles, found 650 relevant
Processed 180000 articles, found 698 relevant
Processed 190000 articles, found 729 relevant
Processed 200000 articles, found 757 relevant
Processed 210000 articles, found 793 relevant
Processed 220000 articles, found 837 relevant

In [52]:
# Create knowledge text
wiki_df['knowledge'] = wiki_df.apply(
    lambda row: f"Title: {row['title']}. {row['text']}", 
    axis=1
)

print(f"Knowledge base size: {len(wiki_df)}")
print(f"\nSample knowledge text:")
print(wiki_df['knowledge'][0][:200])

Knowledge base size: 5000

Sample knowledge text:
Title: Agricultural science. Agricultural science (or agriscience for short) is a broad multidisciplinary field of biology that encompasses the parts of exact, natural, economic and social sciences th


In [53]:
wiki_texts      = wiki_df['knowledge'].tolist()
wiki_embeddings = model_minilm.encode(
    wiki_texts,
    batch_size=64,
    show_progress_bar=True
)

print(f"\n✅ Encoding complete!")
print(f"Embeddings shape: {wiki_embeddings.shape}")

Batches:   0%|          | 0/79 [00:00<?, ?it/s]


✅ Encoding complete!
Embeddings shape: (5000, 384)


In [54]:
import faiss

wiki_embeddings_f32 = wiki_embeddings.astype(np.float32)
faiss.normalize_L2(wiki_embeddings_f32)

wiki_index = faiss.IndexFlatIP(384)
wiki_index.add(wiki_embeddings_f32)

print(f"✅ Wikipedia FAISS index built!")
print(f"Total vectors: {wiki_index.ntotal}")

✅ Wikipedia FAISS index built!
Total vectors: 5000


In [55]:
def retrieve_wiki_context(question, top_k=3):
    # Encode question
    question_emb = model_minilm.encode(
        [question],
        convert_to_tensor=False
    ).astype(np.float32)
    
    # Normalize
    faiss.normalize_L2(question_emb)
    
    # Search Wikipedia index
    scores, indices = wiki_index.search(question_emb, top_k)
    
    # Return relevant articles
    contexts = []
    for idx in indices[0]:
        contexts.append(wiki_df['knowledge'].iloc[idx])
    
    return contexts


In [56]:
question = train_data['prompt'].iloc[0]
print("Question:", question[:80])
print()

contexts = retrieve_wiki_context(question, top_k=3)
print("Retrieved Wikipedia contexts:")
for i, ctx in enumerate(contexts):
    print(f"\nContext {i+1}: {ctx[:200]}")

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationsh

Retrieved Wikipedia contexts:

Context 1: Title: B-theory of time. The B-theory of time, also called the "tenseless theory of time", is one of two positions regarding the temporal ordering of events in the philosophy of time. B-theorists argu

Context 2: Title: Duration (philosophy). Duration (French: la durée) is a theory of time and consciousness posited by the French philosopher Henri Bergson. Bergson sought to improve upon inadequacies he perceive

Context 3: Title: Time in physics. In physics, time is defined by its measurement: time is what a clock reads.  In classical, non-relativistic physics, it is a scalar quantity (often denoted by the symbol ) and,


In [57]:
def predict_wiki_rag(row):
    # Step 1: Retrieve Wikipedia contexts
    contexts = retrieve_wiki_context(row['prompt'], top_k=3)
    
    # Step 2: Combine question + contexts
    combined = row['prompt'] + " " + " ".join(contexts)
    
    # Step 3: Encode combined context
    context_emb = model_minilm.encode(
        combined, convert_to_tensor=True)
    
    # Step 4: Compare each option
    scores = {}
    for opt in all_options:
        opt_emb     = model_minilm.encode(
            row[opt], convert_to_tensor=True)
        scores[opt] = util.cos_sim(
            context_emb, opt_emb).item()
    
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked


In [58]:
row1   = train_data.iloc[0]
result = predict_wiki_rag(row1)

print("Question:", row1['prompt'][:80])
print("Correct answer:", row1['answer'])
print("Wikipedia RAG ranking:", result)
print("Top 3:", ' '.join(result[:3]))

if result[0] == row1['answer']:
    print("CORRECT!")
else:
    print(f"WRONG! Correct at position {result.index(row1['answer'])+1}")

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationsh
Correct answer: B
Wikipedia RAG ranking: ['C', 'D', 'B', 'E', 'A']
Top 3: C D B
WRONG! Correct at position 3


In [59]:
print("Evaluating Wikipedia RAG on 2000 train rows...")

actual_wiki    = train_data['answer'].tolist()
predicted_wiki = train_data.apply(
                    predict_wiki_rag, axis=1).tolist()
predicted_top1_wiki = [p[0] for p in predicted_wiki]

map3_wiki = mapk(actual_wiki, predicted_wiki)
acc_wiki  = accuracy_score(actual_wiki, predicted_top1_wiki)
f1_wiki   = f1_score(actual_wiki, predicted_top1_wiki,
                     average='weighted')

print(f"\nModel 7 Wikipedia RAG Results:")
print(f"MAP@3    : {map3_wiki:.4f}")
print(f"Accuracy : {acc_wiki:.4f}")
print(f"F1 Score : {f1_wiki:.4f}")

Evaluating Wikipedia RAG on 2000 train rows...

Model 7 Wikipedia RAG Results:
MAP@3    : 0.4124
Accuracy : 0.2410
F1 Score : 0.2435


## Lookup + Wikipedia RAG

In [60]:
# Lookup + Wikipedia RAG
def predict_lookup_wiki(row):
    scores = {}
    for opt in all_options:
        opt_text = row[opt]
        if opt_text in answer_text_lookup:
            correct_text = answer_text_lookup[opt_text]
            if opt_text == correct_text:
                scores[opt] = 9999
            else:
                scores[opt] = len(opt_text)
        else:
            # Wikipedia RAG fallback
            contexts    = retrieve_wiki_context(row['prompt'])
            combined    = row['prompt'] + " " + " ".join(contexts)
            context_emb = model_minilm.encode(
                combined, convert_to_tensor=True)
            opt_emb     = model_minilm.encode(
                opt_text, convert_to_tensor=True)
            scores[opt] = util.cos_sim(
                context_emb, opt_emb).item()
    
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked

# Generate submission
predictions_wiki = []
for _, row in test_data.iterrows():
    ranked = predict_lookup_wiki(row)
    predictions_wiki.append(' '.join(ranked[:3]))

submission_wiki         = test_data[['id']].copy()
submission_wiki.columns = ['ID']
submission_wiki['Prediction'] = predictions_wiki
submission_wiki.to_csv('/kaggle/working/submission.csv', index=False)


In [61]:
run = wandb.init(
    entity="21f3001142-iyyiitm",
    project="dlgenai-t226",
    name="model-7-wikipedia-rag",
    config={
        "model"          : "Wikipedia RAG",
        "knowledge_base" : "Wikipedia",
        "kb_size"        : len(wiki_df),
        "encoder"        : "MiniLM",
        "retriever"      : "FAISS",
        "top_k"          : 3,
    }
)
run.log({
    "train_map@3"    : map3_wiki,
    "train_accuracy" : acc_wiki,
    "train_f1"       : f1_wiki,
})
run.finish()


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_014648-q1tvy76v
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-7-wikipedia-rag
wandb: ⭐️ View project at https://wandb.ai/21f3001142-iyyiitm/dlgenai-t226
wandb: 🚀 View run at https://wandb.ai/21f3001142-iyyiitm/dlgenai-t226/runs/q1tvy76v
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: train_accuracy ▁
wandb:       train_f1 ▁
wandb:    train_map@3 ▁
wandb: 
wandb: Run summary:
wandb: train_accuracy 0.241
wandb:       train_f1 0.24346
wandb:    train_map@3 0.41242
wandb: 
wandb: 🚀 View run model-7-wikipedia-rag at: https://wandb.ai/21f3001142-iyyiitm/dlgenai-t226/runs/q1tvy76v
wandb: ⭐️ View project at: https://wandb.ai/21f3001142-iyyiitm/dlgenai-t226
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Fin